# Section 5.6 — Sensitivity to Stages

Reproduces Table 8 from the thesis.  
Runs **EEV, SP, WS** on a **60-cage** synthetic fleet across 2-stage (9 sc.), 3-stage (27 sc.), and 4-stage (81 sc.) scenario trees. DE is not attempted (OOM at 60 cages).

| Config | Stages | Scenarios | Stage slices (months) | EEV fixes |
|--------|--------|-----------|----------------------|-----------|
| 2-stage | 2 | 3² = 9  | 0-29, 30-59           | months 0-29  |
| 3-stage | 3 | 3³ = 27 | 0-19, 20-39, 40-59    | months 0-39  |
| 4-stage | 4 | 3⁴ = 81 | 0-14, 15-29, 30-44, 45-59 | months 0-44 |


In [1]:
import sys
import os
import importlib.util

_here   = os.path.dirname(os.path.abspath('__file__'))
_models = os.path.join(_here, '..', 'models')
sys.path.insert(0, _models)
sys.path.insert(0, _here)

import time
import numpy as np
import pandas as pd

from instance import (
    T, loc_mab, regional_mab,
    units_df as full_units_df,
    temps_bad_12, temps_normal_12, temps_good_12,
    S_normal_v, S_bad_v,
)
from IP import SalmonFarmingMILP

# Load SP classes — 2SP.py / 3SP.py start with digits so use importlib
def _load_ald(filename):
    spec = importlib.util.spec_from_file_location('_sp', os.path.join(_here, filename))
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.AugmentedLagrangianDecomposition

ALD2 = _load_ald('2SP.py')
ALD3 = _load_ald('3SP.py')

# 4-stage SP lives in models/
from SP import AugmentedLagrangianDecomposition as ALD4

print('Imports OK')


Imports OK


In [2]:
# Synthetic 60-cage fleet (5 × 12 cages across 5 locations)

def build_synthetic_fleet(n_target):
    base   = full_units_df.copy()
    n_base = len(base)
    rows   = []
    loc_num   = 1
    remaining = n_target
    while remaining > 0:
        take  = min(n_base, remaining)
        chunk = base.iloc[:take].copy()
        chunk['location'] = f'Loc{loc_num}'
        if loc_num > 1:
            chunk['count'] = float('nan')
            chunk['avg_weight_g'] = float('nan')
        rows.append(chunk)
        remaining -= take
        loc_num   += 1
    df = pd.concat(rows, ignore_index=True)
    loc_mab_multi = {
        f'Location {i + 1}': loc_mab['Location 1']
        for i in range(loc_num - 1)
    }
    return df, loc_mab_multi


units_df_60, loc_mab_60 = build_synthetic_fleet(60)
print(f'Fleet: {len(units_df_60)} cages, {units_df_60["location"].nunique()} locations')
print(units_df_60.groupby('location').size().to_frame('cages'))


Fleet: 60 cages, 5 locations
          cages
location       
Loc1         12
Loc2         12
Loc3         12
Loc4         12
Loc5         12


In [3]:
# Stage configurations

TEMP_MAP = {
    'bad':    temps_bad_12,
    'normal': temps_normal_12,
    'good':   temps_good_12,
}
LABELS = ['bad', 'normal', 'good']

STAGE_CFG = {
    2: {
        'slices':     [list(range(0, 30)), list(range(30, T))],
        'na_months':  set(range(0, 30)),
        'ald_class':  ALD2,
    },
    3: {
        'slices':     [list(range(0, 20)), list(range(20, 40)), list(range(40, T))],
        'na_months':  set(range(0, 40)),
        'ald_class':  ALD3,
    },
    4: {
        'slices':     [list(range(0, 15)), list(range(15, 30)), list(range(30, 45)), list(range(45, T))],
        'na_months':  set(range(0, 45)),
        'ald_class':  ALD4,
    },
}


def _tile(arr, n):
    return np.tile(arr, (n // 12) + 1)[:n]


def build_scenarios_for(n_stages):
    """Build all 3^n_stages (name, temps, S, prob) tuples for the given stage count."""
    slices = STAGE_CFG[n_stages]['slices']
    n_sc   = 3 ** n_stages

    from itertools import product
    scenarios = []
    for combo in product(LABELS, repeat=n_stages):
        name     = '__'.join(f's{i+1}_{sl}' for i, sl in enumerate(combo))
        temps_sc = np.zeros(T)
        S_sc     = np.full(T, S_normal_v)
        for i, (sl, months) in enumerate(zip(combo, slices)):
            prev = combo[i - 1] if i > 0 else None
            surv = S_bad_v if (sl == 'good' and prev == 'good') else S_normal_v
            for mm in months:
                S_sc[mm] = surv
            temps_sc[months] = _tile(TEMP_MAP[sl], len(months))
        scenarios.append((name, temps_sc, S_sc, 1.0 / n_sc))
    return scenarios


In [4]:
# WS helper

def run_ws_for(units_df_sub, loc_mab_sub, n_stages, mip_gap=0.02):
    scenarios = build_scenarios_for(n_stages)
    n_feas = 0
    ws_obj = 0.0
    t0 = time.time()
    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
            scenario_name=sc_name,
        )
        milp.model.Params.OutputFlag = 0
        milp.model.Params.MIPGap = mip_gap
        milp.model.optimize()
        if milp.model.SolCount > 0:
            n_feas += 1
            ws_obj += prob * milp.model.ObjVal
    return ws_obj, time.time() - t0, n_feas


# EEV helper

def run_eev_for(units_df_sub, loc_mab_sub, n_stages, mip_gap=0.02):
    na_months = STAGE_CFG[n_stages]['na_months']

    # Step 1: EV model (expected-parameter deterministic IP)
    temps_exp = np.tile(
        np.array([5, 5, 5, 6, 9, 12, 14, 16.5, 15.5, 13, 10, 7.5]),
        (T // 12) + 1
    )[:T]
    S_exp = np.full(T, (2.0 * S_normal_v + S_bad_v) / 3.0)

    t0 = time.time()
    ev_m = SalmonFarmingMILP(
        units_df=units_df_sub, temps_t=temps_exp, survival_rates=S_exp,
        horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
        density_limit=25.0, verbose=False,
    )
    ev_m.model.Params.OutputFlag = 0
    ev_m.model.Params.MIPGap = mip_gap
    ev_m.model.optimize()
    if ev_m.model.SolCount == 0:
        print('  EV model infeasible')
        return None, time.time() - t0, 0

    # Step 2: Extract decisions for fixed stages
    ev_stk = {
        (u, t): round(ev_m.variables['z'][u, t].X)
        for u in ev_m.U for t in ev_m.Tset if t in na_months
    }
    ev_harv = {
        (u, s, t): round(ev_m.variables['h'][u, s, t].X)
        for u in ev_m.U for s in ev_m.Tset
        for t in ev_m.H_by_us.get((u, s), []) if t in na_months
    }
    ev_hexist = {
        (u, t): round(ev_m.variables['h_exist'][u, t].X)
        for u in ev_m.U_exist for t in ev_m.Tset if t in na_months
    }
    ev_q = {
        (u, s): ev_m.variables['q'][u, s].X
        for u in ev_m.U for s in ev_m.Tset
        if s in na_months and round(ev_m.variables['z'][u, s].X) == 1
    }

    # Step 3: Evaluate across all n_stages scenarios
    scenarios = build_scenarios_for(n_stages)
    n_feas = 0
    eev_obj = 0.0
    feas_prob = 0.0

    for sc_name, temps_sc, S_sc, prob in scenarios:
        milp = SalmonFarmingMILP(
            units_df=units_df_sub, temps_t=temps_sc, survival_rates=S_sc,
            horizon_months=T, loc_mab=loc_mab_sub, regional_mab=regional_mab,
            scenario_name=sc_name, disable_economic_presolve=True,
        )
        model = milp.model
        model.update()
        for (u, t), val in ev_stk.items():
            v = model.getVarByName(f'z[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s, t), val in ev_harv.items():
            v = model.getVarByName(f'h[{u},{s},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, t), val in ev_hexist.items():
            v = model.getVarByName(f'h_exist[{u},{t}]')
            if v is not None:
                v.LB = val; v.UB = val
        for (u, s), val in ev_q.items():
            v = model.getVarByName(f'q[{u},{s}]')
            if v is not None:
                v.LB = val; v.UB = val
        model.Params.OutputFlag = 0
        model.Params.MIPGap = mip_gap
        model.update()
        model.optimize()
        if model.SolCount > 0:
            n_feas += 1
            feas_prob += prob
            eev_obj += prob * model.ObjVal

    eev_normalized = eev_obj / feas_prob if feas_prob > 0 else 0.0
    return eev_normalized, time.time() - t0, n_feas


In [ ]:
# Main experiment: {2, 3, 4} stages × {EEV, SP, WS}  —  60 cages, mip_gap=2%

MIP_GAP = 0.02
results = []

for n_stages in [2, 3, 4]:
    n_sc = 3 ** n_stages
    print(f'\n{"="*70}')
    print(f'  {n_stages}-stage tree  ({n_sc} scenarios)  —  60 cages')
    print(f'{"="*70}')

    # EEV
    print(f'\n--- EEV ({n_stages}-stage) ---')
    eev_obj, eev_time, eev_feas = run_eev_for(
        units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
    )
    print(f'  EEV = {eev_obj/1e6:.0f} MNOK  |  {eev_time:.0f}s  |  {eev_feas}/{n_sc} feasible')

    # SP
    print(f'\n--- SP ({n_stages}-stage) ---')
    ALD = STAGE_CFG[n_stages]['ald_class']
    ald = ALD(
        units_df=units_df_60, loc_mab=loc_mab_60, regional_mab=regional_mab,
        T=T, mip_gap=MIP_GAP,
        temps_bad=temps_bad_12, temps_normal=temps_normal_12, temps_good=temps_good_12,
    )
    ald.build()
    ald.solve()
    sp_obj  = ald.eval_obj
    sp_time = ald.total_time
    print(f'  SP  = {sp_obj/1e6:.0f} MNOK  |  {sp_time:.0f}s  |  {ald.n_feasible}/{ald.n_scenarios} feasible')

    # WS
    print(f'\n--- WS ({n_stages}-stage) ---')
    ws_obj, ws_time, ws_feas = run_ws_for(
        units_df_60, loc_mab_60, n_stages, mip_gap=MIP_GAP
    )
    print(f'  WS  = {ws_obj/1e6:.0f} MNOK  |  {ws_time:.0f}s  |  {ws_feas}/{n_sc} feasible')

    vss_sp  = (sp_obj - eev_obj) / 1e6
    evpi_sp = (ws_obj - sp_obj)  / 1e6
    print(f'  VSS_SP={vss_sp:.0f} MNOK   EVPI_SP={evpi_sp:.0f} MNOK')

    results.append({
        'stages':         n_stages,
        'scenarios':      n_sc,
        'EEV feas':       f'{eev_feas}/{n_sc}',
        'EEV [MNOK]':     round(eev_obj / 1e6, 0),
        'SP [MNOK]':      round(sp_obj  / 1e6, 0),
        'WS [MNOK]':      round(ws_obj  / 1e6, 0),
        'VSS_SP [MNOK]':  round(vss_sp,  0),
        'EVPI_SP [MNOK]': round(evpi_sp, 0),
        'EEV time [s]':   round(eev_time,  0),
        'SP time [s]':    round(sp_time,   0),
        'WS time [s]':    round(ws_time,   0),
        'MIP gap':        MIP_GAP,
    })



  2-stage tree  (9 scenarios)  —  60 cages

--- EEV (2-stage) ---
Set parameter Username
Set parameter LicenseID to value 2786519
Academic license - for non-commercial use only - expires 2027-03-03


In [ ]:
# Results table (Table 8)

df_results = pd.DataFrame(results).set_index('stages')
display(df_results)
